# Summary: the three models, side by side

**Course**: ICS554 Natural Language Processing · Ashesi University  
**Project**: Prosit 1 (Ankora AI Research Lab)  
**Objective**: Collect the final numbers for the report and slides straight from the result files, so nothing is typed in by hand.

The repository holds three separate models:

| | Model | Report | Language and data | Code | Results |
|---|---|---|---|---|---|
| 1 | n-gram (interpolated Kneser-Ney) | Section B | Ewe sentences | `src/section_b_ngram/` | `results/section_b_ngram/` |
| 2 | LSTM, trained from scratch | Section B, Question 2 | the same Ewe tokens as the BPE n-gram | `src/section_b_lstm/` | `results/section_b_lstm/` |
| 3 | distilgpt2 + LoRA | Section C | English agricultural Q&A | `src/section_c_llm/` | `results/section_c_llm/` |

Models 1 and 2 are scored per word on the same Ewe test sentences, so their perplexities compare. Model 3 is scored on English text with its own tokenizer: its perplexities do not compare with Section B's.

In [ ]:
import sys
import json
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

RESULTS = REPO_ROOT / "results"
ngram = json.loads((RESULTS / "section_b_ngram" / "unified.json").read_text())
lstm = json.loads((RESULTS / "section_b_lstm" / "lstm_vs_ngram.json").read_text())
lora = json.loads((RESULTS / "section_c_llm" / "lora_results.json").read_text())

## 1. Section B: n-gram, best order per tokenizer (unified corpus)
Per-token perplexity cannot be compared across tokenizers; per-word perplexity can (same text, same number of words, and every `<unk>` pays the cost of spelling the word).

In [ ]:
rows = {}
for tok, best in ngram["best_order_by_val"].items():
    row = next(r for r in ngram["results"][tok] if r["order"] == best["order"])
    rows[tok] = {"best N (val)": best["order"], "val PPL": row["val_perplexity"], "test PPL / token": row["perplexity"],
                 "test PPL / word": row["per_word_perplexity"], "vocab": row["vocab_size"], "OOV %": row["oov_rate_pct"]}
print(ngram["dataset"], "|", ngram["smoothing"])
pd.DataFrame.from_dict(rows, orient="index")

## 2. Section B, Question 2: LSTM against the n-gram, on the same tokens
Both models read the same BPE tokens (150 merges) and are scored on the same test sentences, with the same spelling charge for `<unk>`. The LSTM runs three seeds; a model wins only if every seed falls on the same side of the n-gram.

In [ ]:
rows = {}
for e in lstm.values():
    runs, kn = e["runs"], e["kn_bpe"]["per_word_perplexity"]
    minutes = [r["train_seconds"] / 60 for r in runs]
    rows[e["dataset"]] = {
        "train sentences": e["train_sentences"],
        "LSTM parameters": runs[0]["params"],
        "LSTM per word (mean)": e["lstm_per_word_mean"],
        "LSTM range": f'{e["lstm_per_word_min"]} to {e["lstm_per_word_max"]}',
        "n-gram per word": kn,
        "winner": "n-gram" if kn < e["lstm_per_word_min"] else "LSTM" if kn > e["lstm_per_word_max"] else "no clear winner",
        "LSTM minutes per seed": f"{min(minutes):.1f} to {max(minutes):.1f}",
    }
pd.DataFrame.from_dict(rows, orient="index")

## 3. Section C: distilgpt2 + LoRA (English; not comparable with Section B)

In [ ]:
print(lora["split_sizes"], "| test questions seen in train/val:", lora["test_questions_seen_in_train_or_val"])
pd.DataFrame({name: {m: r[m] for m in ("full_ppl", "answer_ppl", "wikitext_ppl")} for name, r in lora["results"].items()}).T